# Module X2: exercises  (solutions)

These use `customer_churn.csv`: 400 customers, `tenure_months`, `churned` (1 = churned, 0 = censored), `plan` and `monthly_spend`.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**Kaplan–Meier by hand.** Write `kaplan_meier(time, event)` returning a DataFrame with columns `t` (distinct **event** times), `n_risk` (customers with time $\ge t$), `d` (events at $t$) and `S`
($\hat S=\prod(1-d_i/n_i)$). Apply it to all customers as `km`.

In [ ]:
import math

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar
from statsmodels.duration.hazard_regression import PHReg
from statsmodels.duration.survfunc import SurvfuncRight

churn = pd.read_csv(DATA / "customer_churn.csv")
t, e = churn["tenure_months"].to_numpy(), churn["churned"].to_numpy()

def kaplan_meier(time, event):
    time, event = np.asarray(time, dtype=float), np.asarray(event)
    rows, S = [], 1.0
    for u in np.unique(time[event == 1]):
        n_i = int((time >= u).sum())
        d_i = int(((time == u) & (event == 1)).sum())
        S *= 1 - d_i / n_i
        rows.append((u, n_i, d_i, S))
    return pd.DataFrame(rows, columns=["t", "n_risk", "d", "S"])

km = kaplan_meier(t, e)

In [ ]:
tiny = kaplan_meier([1, 2, 3, 4], [1, 0, 1, 1])
assert list(tiny["t"]) == [1, 3, 4] and list(tiny["n_risk"]) == [4, 2, 1], "a censored customer leaves the risk set after its time"
assert np.allclose(tiny["S"], [0.75, 0.375, 0.0])
ref = SurvfuncRight(t, e)
assert np.allclose(km["S"], ref.surv_prob) and np.allclose(km["t"], ref.surv_times)
assert km["n_risk"].iloc[0] == 400 and km["d"].sum() == 302
print(f"✅ Correct! {len(km)} event times; the curve ends at S = {km['S'].iloc[-1]:.3f}")

## Exercise 2

**Read the curve.** Write `survival_at(km, time)` (the last `S` at or before `time`; 1.0 if no event yet) and `median_survival(km)` (the first `t` where `S <= 0.5`).
Using your `kaplan_meier` on each plan, store `s12_basic`, `s12_premium` (survival at 12 months) and `med_basic`, `med_premium`.

In [ ]:
def survival_at(km, time):
    before = km[km["t"] <= time]
    return 1.0 if before.empty else before["S"].iloc[-1]

def median_survival(km):
    below = km[km["S"] <= 0.5]
    return below["t"].iloc[0] if len(below) else np.nan

by_plan = {p: kaplan_meier(d["tenure_months"], d["churned"]) for p, d in churn.groupby("plan")}
s12_basic, s12_premium = survival_at(by_plan["basic"], 12), survival_at(by_plan["premium"], 12)
med_basic, med_premium = median_survival(by_plan["basic"]), median_survival(by_plan["premium"])

In [ ]:
k = kaplan_meier([1, 2, 3, 4], [1, 1, 1, 1])
assert survival_at(k, 0.5) == 1.0 and survival_at(k, 2.5) == 0.5 and median_survival(k) == 2
assert abs(s12_basic - 0.3617) < 1e-3 and abs(s12_premium - 0.5636) < 1e-3
assert abs(med_basic - 7.80) < 0.01 and abs(med_premium - 14.47) < 0.01
print(f"✅ Correct! Median survival {med_basic:.1f} months (basic) vs {med_premium:.1f} (premium); {s12_basic:.0%} vs {s12_premium:.0%} remain after a year")

## Exercise 3

**Greenwood interval.** Greenwood's formula gives $\operatorname{SE}[\hat S(t)]=\hat S(t)\sqrt{\sum_{t_i\le t}\frac{d_i}{n_i(n_i-d_i)}}$. Write `greenwood_ci(km, time)` returning the
**log-log 95% CI** $\big(\hat S^{\exp(+1.96\,s)},\,\hat S^{\exp(-1.96\,s)}\big)$ with $s=\dfrac{\sqrt{\sum d_i/(n_i(n_i-d_i))}}{|\ln\hat S|}$ for survival at `time`.
Apply it to all customers at 12 months as `ci12`.

In [ ]:
def greenwood_ci(km, time):
    used = km[km["t"] <= time]
    S = used["S"].iloc[-1]
    gw = (used["d"] / (used["n_risk"] * (used["n_risk"] - used["d"]))).sum()
    s = math.sqrt(gw) / abs(math.log(S))
    return S ** math.exp(1.96 * s), S ** math.exp(-1.96 * s)

ci12 = greenwood_ci(km, 12)

In [ ]:
lo, hi = ci12
assert 0 < lo < survival_at(km, 12) < hi < 1
assert abs(lo - 0.3958) < 1e-3 and abs(hi - 0.4930) < 1e-3, ci12
print(f"✅ Correct! S(12) = {survival_at(km, 12):.3f}, 95% CI ({lo:.3f}, {hi:.3f})")

## Exercise 4

**Log-rank test.** Write `logrank(time, event, group)` returning `(chi2, p, observed, expected)` for the **first group in sorted order** (`basic` for plans). At each event time $u$: $n$ at risk, $n_1$ at risk in group 1,
$d$ events, $d_1$ in group 1; $O=\sum d_1$, $E=\sum d\,n_1/n$, $V=\sum d\frac{n_1}{n}\big(1-\frac{n_1}{n}\big)\frac{n-d}{n-1}$, $\chi^2=(O-E)^2/V$ with 1 df.
Apply it to plans as `lr`.

In [ ]:
def logrank(time, event, group):
    time, event, group = np.asarray(time, dtype=float), np.asarray(event), np.asarray(group)
    g1 = group == np.unique(group)[0]
    O = E = V = 0.0
    for u in np.unique(time[event == 1]):
        at_risk = time >= u
        n, n1 = at_risk.sum(), (at_risk & g1).sum()
        d, d1 = ((time == u) & (event == 1)).sum(), ((time == u) & (event == 1) & g1).sum()
        O += d1
        E += d * n1 / n
        if n > 1:
            V += d * (n1 / n) * (1 - n1 / n) * (n - d) / (n - 1)
    chi2 = (O - E) ** 2 / V
    return chi2, stats.chi2.sf(chi2, 1), O, E

lr = logrank(t, e, churn["plan"])

In [ ]:
from statsmodels.duration.survfunc import survdiff
ref = survdiff(t, e, churn["plan"].to_numpy())
assert abs(lr[0] - ref[0]) < 1e-6 and abs(lr[1] - ref[1]) < 1e-12
assert abs(lr[2] - 201) < 1e-9 and abs(lr[3] - 154.9) < 0.1
print(f"✅ Correct! chi2 = {lr[0]:.2f}, p = {lr[1]:.1e}: {lr[2]:.0f} basic customers churned versus {lr[3]:.0f} expected")

## Exercise 5

**Exponential rates with censoring.** Write `exponential_rate(time, event)` $=$ (number of events) / (total time at risk). Compute `rate_basic`, `rate_premium`, the `rate_ratio` (basic / premium) and `ci_basic`,
the 95% CI for the basic rate: $\hat\lambda\,e^{\pm1.96/\sqrt d}$ where $d$ is the number of basic-plan events.

In [ ]:
def exponential_rate(time, event):
    return np.sum(event) / np.sum(time)

basic_df = churn[churn["plan"] == "basic"]
premium_df = churn[churn["plan"] == "premium"]
rate_basic = exponential_rate(basic_df["tenure_months"], basic_df["churned"])
rate_premium = exponential_rate(premium_df["tenure_months"], premium_df["churned"])
rate_ratio = rate_basic / rate_premium
half = 1.96 / math.sqrt(basic_df["churned"].sum())
ci_basic = (rate_basic * math.exp(-half), rate_basic * math.exp(half))

In [ ]:
assert abs(exponential_rate([1, 2, 3], [1, 0, 1]) - 2 / 6) < 1e-12, "censored time counts in the denominator but not the numerator"
assert abs(rate_basic - 0.08891) < 1e-4 and abs(rate_premium - 0.04619) < 1e-4
assert abs(rate_ratio - 1.9249) < 1e-3
assert abs(ci_basic[0] - 0.0774) < 1e-3 and abs(ci_basic[1] - 0.1021) < 1e-3
print(f"✅ Correct! Basic churn rate {rate_basic:.3f}/month (95% CI {ci_basic[0]:.3f}-{ci_basic[1]:.3f}); rate ratio {rate_ratio:.2f}")

## Exercise 6

**Cox regression.** Add a 0/1 column `basic` and fit `PHReg.from_formula("tenure_months ~ basic + monthly_spend", data, status=e).fit()` as `cox`.
Store `hr_basic` (hazard ratio for basic vs premium), `hr_basic_ci` (a tuple) and `hr_spend10`: the hazard ratio for **$10 more** monthly spend, $\exp(10\,\beta_{\text{spend}})$.

In [ ]:
data = churn.assign(basic=(churn["plan"] == "basic").astype(int))
cox = PHReg.from_formula("tenure_months ~ basic + monthly_spend", data, status=e).fit()
hr_basic = float(np.exp(cox.params[0]))
ci = np.exp(cox.conf_int()[0])
hr_basic_ci = (float(ci[0]), float(ci[1]))
hr_spend10 = float(np.exp(10 * cox.params[1]))

In [ ]:
assert abs(hr_basic - 1.905) < 0.005 and abs(hr_basic_ci[0] - 1.497) < 0.01 and abs(hr_basic_ci[1] - 2.425) < 0.01
assert abs(hr_spend10 - 0.853) < 0.005, hr_spend10
assert hr_basic_ci[0] > 1, "the whole interval is above 1: basic customers have a clearly higher hazard"
print(f"✅ Correct! Basic vs premium HR = {hr_basic:.2f} (95% CI {hr_basic_ci[0]:.2f}-{hr_basic_ci[1]:.2f}); $10 more spend: HR = {hr_spend10:.2f}")

## Exercise 7

**The partial likelihood by hand.** Write `cox_beta(time, event, x)` for **one covariate** `x`: maximise $\sum_{i:\delta_i=1}\Big[\beta x_i-\ln\sum_{j:\,t_j\ge t_i}e^{\beta x_j}\Big]$ over $\beta$
(minimise its negative with `scipy.optimize.minimize_scalar(..., bounds=(-3, 3), method="bounded")`). Apply it to the 0/1 `basic` indicator as `beta_hand`.

In [ ]:
def cox_beta(time, event, x):
    time, event, x = np.asarray(time, dtype=float), np.asarray(event), np.asarray(x, dtype=float)

    def neg_pll(beta):
        ll = 0.0
        for i in np.where(event == 1)[0]:
            risk = time >= time[i]
            ll += beta * x[i] - np.log(np.exp(beta * x[risk]).sum())
        return -ll

    return minimize_scalar(neg_pll, bounds=(-3, 3), method="bounded").x

beta_hand = cox_beta(t, e, data["basic"])

In [ ]:
simple = PHReg.from_formula("tenure_months ~ basic", data, status=e).fit()
assert abs(beta_hand - simple.params[0]) < 1e-3, (beta_hand, simple.params[0])
assert abs(math.exp(beta_hand) - 1.911) < 0.005
print(f"✅ Correct! beta = {beta_hand:.4f} -> HR = {math.exp(beta_hand):.2f}, identical to the library's partial-likelihood fit")

## Exercise 8

**Restricted mean survival time.** The area under the Kaplan–Meier step curve up to a horizon $\tau$ is the average number of months a customer stays within the first $\tau$ months.
Write `rmst(km, horizon)`: starting from $S=1$ at $t=0$, add $S\times$(width of each interval up to `horizon`). Compute `rmst24` for all customers.

In [ ]:
def rmst(km, horizon):
    times = np.r_[0.0, km["t"].to_numpy()]
    surv = np.r_[1.0, km["S"].to_numpy()]
    total = 0.0
    for i in range(len(times)):
        if times[i] >= horizon:
            break
        end = min(times[i + 1], horizon) if i + 1 < len(times) else horizon
        total += surv[i] * (end - times[i])
    return total

rmst24 = rmst(km, 24)

In [ ]:
k = kaplan_meier([2, 4], [1, 1])                  # S = 1 until 2, 0.5 from 2 to 4, 0 afterwards
assert abs(rmst(k, 4) - (1 * 2 + 0.5 * 2)) < 1e-12 and abs(rmst(k, 3) - (1 * 2 + 0.5 * 1)) < 1e-12
assert abs(rmst24 - 11.76) < 0.02, rmst24
print(f"✅ Correct! Within 24 months a customer stays {rmst24:.1f} months on average (restricted mean survival time)")

## Exercise 9

**Which statement is true?** Set `answer` to a letter.

* **a)** A censored customer should be dropped from the analysis because we don't know when they would have churned.
* **b)** A censored customer contributes the information that they were still a customer at their last observation, and stays in the risk set until then.
* **c)** Censored times should be treated as churn times for the Kaplan–Meier calculation.
* **d)** A hazard ratio of 2 means a basic customer will, on average, stay exactly half as long, whatever the time scale.

In [ ]:
answer = "b"

In [ ]:
assert answer == "b", "Censoring means 'at least this long': it contributes to the risk set until the censoring time"
print("✅ Correct!")